# Save and load objects

ffsim can save many of its objects to disk and load them back later. [ffsim.to_json](../api/stubs/ffsim.to_json.rst#ffsim.to_json) writes an object to a JSON file, which can optionally be compressed, and [ffsim.from_json](../api/stubs/ffsim.from_json.rst#ffsim.from_json) reads it back. The objects that can be saved are:

- Molecular data: [ffsim.MolecularData](../api/stubs/ffsim.MolecularData.rst#ffsim.MolecularData)
- Hamiltonians, such as [ffsim.MolecularHamiltonian](../api/stubs/ffsim.MolecularHamiltonian.rst#ffsim.MolecularHamiltonian) and [ffsim.DoubleFactorizedHamiltonian](../api/stubs/ffsim.DoubleFactorizedHamiltonian.rst#ffsim.DoubleFactorizedHamiltonian)
- Ansatz operators, such as [ffsim.UCJOpSpinBalanced](../api/stubs/ffsim.UCJOpSpinBalanced.rst#ffsim.UCJOpSpinBalanced) and [ffsim.UCCSDOpRestrictedReal](../api/stubs/ffsim.UCCSDOpRestrictedReal.rst#ffsim.UCCSDOpRestrictedReal)
- [ffsim.StateVector](../api/stubs/ffsim.StateVector.rst#ffsim.StateVector) and [ffsim.ReducedDensityMatrix](../api/stubs/ffsim.ReducedDensityMatrix.rst#ffsim.ReducedDensityMatrix)
- [ffsim.FermionOperator](../api/stubs/ffsim.FermionOperator.rst#ffsim.FermionOperator)

The file records the object's type, so `from_json` gives back an object of the original class. NumPy arrays keep their exact dtype and shape, so loading a saved object restores its data bit for bit.

## Save and load molecular data

The following code cell runs a Hartree-Fock calculation on the nitrogen molecule and stores the results in a `MolecularData` object.

In [1]:
import json
import pathlib
import tempfile

import numpy as np
import pyscf

import ffsim

rng = np.random.default_rng(12345)

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, -0.55)], ["N", (0, 0, 0.55)]],
    basis="sto-6g",
    symmetry="Dooh",
    verbose=0,
)

# Run Hartree-Fock, freezing the two 1s core orbitals
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf, active_space=range(2, mol.nao_nr()))

# Create a temporary directory to hold the files written in this guide
tmp_dir = tempfile.TemporaryDirectory()
directory = pathlib.Path(tmp_dir.name)

Pass the object and a file path to `to_json` to save it, and pass the path to `from_json` to load it.

In [2]:
path = directory / "n2.json"
ffsim.to_json(mol_data, path)

loaded = ffsim.from_json(path)

print(f"Type: {type(loaded).__name__}")
print(f"Hartree-Fock energy: {loaded.hf_energy}")
print(
    "One-body integrals match: "
    f"{np.array_equal(loaded.one_body_integrals, mol_data.one_body_integrals)}"
)
print(
    "Two-body integrals match: "
    f"{np.array_equal(loaded.two_body_integrals, mol_data.two_body_integrals)}"
)

Type: MolecularData
Hartree-Fock energy: -108.5424713088927
One-body integrals match: True
Two-body integrals match: True


The loaded object behaves just like the original. For example, its `hamiltonian` property returns the molecular Hamiltonian, which we can use to compute the Hartree-Fock energy from the Hartree-Fock state.

In [3]:
hamiltonian = ffsim.linear_operator(
    loaded.hamiltonian, norb=loaded.norb, nelec=loaded.nelec
)
hf_state = ffsim.hartree_fock_state(loaded.norb, loaded.nelec)
hf_energy = np.vdot(hf_state, hamiltonian @ hf_state).real
print(f"Hartree-Fock energy: {hf_energy}")

Hartree-Fock energy: -108.54247130889264


## Compression

Files can be compressed with gzip, bz2, or lzma. When saving, `to_json` picks the compression algorithm from the file extension: `.gz` for gzip, `.bz2` for bz2, and `.xz` or `.lzma` for lzma. Any other extension, such as `.json`, means no compression. When loading, `from_json` detects the compression algorithm from the file's contents, so you don't have to specify it. To override this behavior, pass the `compression` argument, which can be `"gzip"`, `"bz2"`, `"lzma"`, or `None` for no compression.

The following code cell saves the molecular data with each compression algorithm and prints the file sizes.

In [4]:
for extension in [".json", ".json.gz", ".json.bz2", ".json.xz"]:
    path = directory / f"n2{extension}"
    ffsim.to_json(mol_data, path)
    loaded = ffsim.from_json(path)
    assert np.array_equal(loaded.two_body_integrals, mol_data.two_body_integrals)
    print(f"{extension:9} {path.stat().st_size:6} bytes")

.json      16648 bytes
.json.gz    4119 bytes
.json.bz2   4393 bytes
.json.xz    4168 bytes


## Save and load other objects

Other objects are saved and loaded the same way. Here, we save a random spin-balanced UCJ operator and a fermion operator.

In [5]:
ucj_op = ffsim.random.random_ucj_op_spin_balanced(
    mol_data.norb, n_reps=2, with_final_orbital_rotation=True, seed=rng
)
path = directory / "ucj_op.json.gz"
ffsim.to_json(ucj_op, path)
loaded_ucj_op = ffsim.from_json(path)

op = ffsim.FermionOperator(
    {
        (ffsim.cre_a(0), ffsim.des_a(1)): 0.5,
        (ffsim.cre_a(1), ffsim.des_a(0)): 0.5,
        (ffsim.cre_b(1), ffsim.des_b(0), ffsim.cre_a(0), ffsim.des_a(1)): 1 - 1j,
    }
)
path = directory / "op.json"
ffsim.to_json(op, path)
loaded_op = ffsim.from_json(path)

print(f"UCJ operators equal: {ffsim.approx_eq(loaded_ucj_op, ucj_op)}")
print(f"Fermion operators equal: {loaded_op == op}")

UCJ operators equal: True
Fermion operators equal: True


## Store objects in your own JSON files

`to_json` writes a single object per file. To store objects alongside other data, such as the results of an experiment, use [ffsim.to_json_dict](../api/stubs/ffsim.to_json_dict.rst#ffsim.to_json_dict). It converts an object to a dictionary that the standard `json` module can write, and that you can nest inside a larger dictionary. After reading the file, pass the dictionary to [ffsim.from_json_dict](../api/stubs/ffsim.from_json_dict.rst#ffsim.from_json_dict) to reconstruct the object.

The following code cell computes the energy of the UCJ ansatz state and saves it in one file together with the operator and the molecular data.

In [6]:
reference_state = ffsim.hartree_fock_state(mol_data.norb, mol_data.nelec)
ansatz_state = ffsim.apply_unitary(
    reference_state, ucj_op, norb=mol_data.norb, nelec=mol_data.nelec
)
energy = np.vdot(ansatz_state, hamiltonian @ ansatz_state).real

record = {
    "energy": energy,
    "ansatz": ffsim.to_json_dict(ucj_op),
    "molecular_data": ffsim.to_json_dict(mol_data),
}
path = directory / "result.json"
path.write_text(json.dumps(record))

record = json.loads(path.read_text())
loaded_ucj_op = ffsim.from_json_dict(record["ansatz"])
loaded_mol_data = ffsim.from_json_dict(record["molecular_data"])

print(f"Energy: {record['energy']}")
print(f"UCJ operators equal: {ffsim.approx_eq(loaded_ucj_op, ucj_op)}")
print(f"Hartree-Fock energy: {loaded_mol_data.hf_energy}")

Energy: -105.41553373080998
UCJ operators equal: True
Hartree-Fock energy: -108.5424713088927


We're done with the files, so we delete the temporary directory.

In [7]:
tmp_dir.cleanup()

## Files written by older versions of ffsim

ffsim 0.0.86 and earlier could only save `MolecularData`, using its `to_json` method. That method and its `from_json` counterpart are now deprecated, and you should use `ffsim.to_json` and `ffsim.from_json` instead. `ffsim.from_json` can still read files written by those older versions, whether or not they're compressed. Those files stored arrays as nested lists without their dtypes, so `from_json` infers each array's dtype from its values.